In [1]:
!pip uninstall -y torchvision -q
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 138.0 MB/s eta 0:00:00


In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

model = model.to(device)
model.eval()

print("Model:", MODEL_NAME)
print("Device:", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda


In [32]:
context = """
Bir elektrik motorunda rulmanlar, rotorun düzgün ve düşük
sürtünmeyle dönmesini sağlar.

Rulman arızalarının yaygın nedenleri arasında yetersiz
yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve
hatalı montaj bulunur.

Rulman aşınması arttığında motor titreşiminde ve çalışma
sıcaklığında artış görülebilir.

Titreşim analizi, sıcaklık ölçümü ve akustik izleme,
rulman arızalarının erken tespiti için kullanılabilir.

Erken tespit edilmeyen ciddi rulman arızaları motorun
performansının düşmesine ve sonunda tamamen durmasına
neden olabilir.
"""

In [33]:
def ask_question(
    context,
    question,
    max_new_tokens=100
):

    messages = [
        {
            "role": "system",
            "content": (
                "Answer the question using only the provided context. "
                "Do not use outside knowledge. "
                "If the answer cannot be found in the context, reply exactly: "
                "'Bu bilgi verilen kaynakta bulunmuyor.' "
                "If the answer exists in the context, answer briefly in Turkish."
            )
        },
        {
            "role": "user",
            "content": f"""
KAYNAK:
{context}

SORU:
{question}
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][
        input_length:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [34]:
question = "Rulman arızalarının yaygın nedenleri nelerdir?"

answer = ask_question(
    context,
    question
)

print(answer)

yetersiz yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj


In [35]:
question = "Rulman aşınması arttığında hangi belirtiler görülebilir?"

answer = ask_question(
    context,
    question
)

print(answer)

Titreşim analizi, sıcaklık ölçümü ve akustik izleme, rulman arızalarının erken tespiti için kullanılabilir.


In [36]:
question = "Rulman arızalarının erken tespiti için hangi yöntemler kullanılabilir?"

answer = ask_question(
    context,
    question
)

print(answer)

Titreşim analizi, sıcaklık ölçümü ve akustik izleme, rulman arızalarının erken tespiti için kullanılabilir.


In [37]:
question = "Motor kaç amper çeker?"

answer = ask_question(
    context,
    question
)

print(answer)

Bu bilgi verilen kaynakta bulunmuyor.


In [38]:
question = "Motorun nominal gerilimi kaç volttur?"

answer = ask_question(
    context,
    question
)

print(answer)

Bu bilgi verilen kaynakta bulunmuyor.


In [39]:
def qa_demo(
    context,
    question
):

    answer = ask_question(
        context,
        question
    )

    print("=" * 70)

    print("SORU:")
    print(question)

    print("\nCEVAP:")
    print(answer)

    print("=" * 70)

In [40]:
questions = [

    "Rulmanın motordaki görevi nedir?",

    "Rulman arızalarının yaygın nedenleri nelerdir?",

    "Rulman aşınması arttığında hangi belirtiler görülebilir?",

    "Rulman arızalarının erken tespiti için hangi yöntemler kullanılabilir?",

    "Erken tespit edilmeyen ciddi rulman arızaları neye neden olabilir?",

    "Motor kaç amper çeker?",

    "Motorun nominal gerilimi kaç volttur?"
]

In [41]:
for question in questions:

    qa_demo(
        context,
        question
    )

SORU:
Rulmanın motordaki görevi nedir?

CEVAP:
Düzgün ve düşük sürtünmeyle rotondan hızla dönmesini sağlar.
SORU:
Rulman arızalarının yaygın nedenleri nelerdir?

CEVAP:
yetersiz yağlama, yanlış hizalama, aşırı mekanik yük, kirlenme ve hatalı montaj
SORU:
Rulman aşınması arttığında hangi belirtiler görülebilir?

CEVAP:
Titreşim analizi, sıcaklık ölçümü ve akustik izleme, rulman arızalarının erken tespiti için kullanılabilir.
SORU:
Rulman arızalarının erken tespiti için hangi yöntemler kullanılabilir?

CEVAP:
Titreşim analizi, sıcaklık ölçümü ve akustik izleme, rulman arızalarının erken tespiti için kullanılabilir.
SORU:
Erken tespit edilmeyen ciddi rulman arızaları neye neden olabilir?

CEVAP:
Motorun performansının düşmesine ve sonunda tamamen durmasına neden olabilir.
SORU:
Motor kaç amper çeker?

CEVAP:
Bu bilgi verilen kaynakta bulunmuyor.
SORU:
Motorun nominal gerilimi kaç volttur?

CEVAP:
Bu bilgi verilen kaynakta bulunmuyor.


In [42]:
context_tokens = tokenizer.encode(
    context,
    add_special_tokens=False
)

print(
    "Kaynak token sayısı:",
    len(context_tokens)
)

Kaynak token sayısı: 183


In [43]:
question = "Rulman arızalarının yaygın nedenleri nelerdir?"

question_tokens = tokenizer.encode(
    question,
    add_special_tokens=False
)

print(
    "Soru token sayısı:",
    len(question_tokens)
)

Soru token sayısı: 15
